# RSNA Knee — Week 1 EDAPurpose: answer the five questions that determine the entire strategy. Run top to bottom on a **CPU** notebook (do not waste GPU quota on this).1. How many studies have gold labels?2. What is the per-label prevalence?3. Can we recover the institution/site from DICOM metadata?4. Can we recover laterality (left/right knee)?5. Are the reports structured, and what languages?Plus: DICOM transfer-syntax survey and a decode-speed benchmark, which set the budget for the caching pipeline.

In [ ]:
import os, sys, glob, json, time, warnings, collections
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
if not os.path.isdir(ROOT):
    # fall back: hunt for the folder that actually contains train.csv
    hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if hits:
        ROOT = os.path.dirname(hits[0])
    else:
        print("Could not locate train.csv under /kaggle/input")
print("ROOT =", ROOT)
print()
for f in sorted(os.listdir(ROOT)):
    p = os.path.join(ROOT, f)
    if os.path.isdir(p):
        print(f"  [dir ] {f}/  ({len(os.listdir(p))} entries)")
    else:
        print(f"  [file] {f}  ({os.path.getsize(p)/1e6:.1f} MB)")


## Q1 — How many studies actually have gold labels?This is the single most important number in the competition.The overview says *"only a small subset of training studies carry per-condition labels"*. If that subset is ~2,000 studies you can train a decent supervised model directly and use pseudo-labels as a bonus. If it's ~400, the report-extraction pipeline **is** the competition and almost all your effort belongs there.We detect label columns automatically rather than hardcoding names, because the exact spelling (`Baker's` vs `Bakers`, spaces vs underscores) isn't guaranteed.

In [ ]:
train = pd.read_csv(f"{ROOT}/train.csv")print("train.csv shape:", train.shape)print("columns:", list(train.columns))print()META_COLS = {"StudyInstanceUID", "PatientSex", "Report"}label_cols = [c for c in train.columns if c not in META_COLS]print(f"Detected {len(label_cols)} label columns:")print(label_cols)print()# A study is "gold labelled" if its label cells are populated (not NaN).nn = train[label_cols].notna()full  = (nn.sum(axis=1) == len(label_cols)).sum()none_ = (nn.sum(axis=1) == 0).sum()part  = len(train) - full - none_print(f"studies total          : {len(train):,}")print(f"  fully labelled       : {full:,}  ({full/len(train):.1%})   <-- GOLD SET")print(f"  partially labelled   : {part:,}  ({part/len(train):.1%})")print(f"  no labels at all     : {none_:,}  ({none_/len(train):.1%})  <-- needs pseudo-labels")print()if "Report" in train.columns:    has_rep = train["Report"].notna() & (train["Report"].astype(str).str.strip() != "")    print(f"studies with a report  : {has_rep.sum():,}  ({has_rep.mean():.1%})")    print(f"  gold AND report      : {(has_rep & (nn.sum(axis=1)==len(label_cols))).sum():,}  <-- extractor train/val set")    print(f"  report, NO gold      : {(has_rep & (nn.sum(axis=1)==0)).sum():,}  <-- extractor inference target")    print(f"  NO report, no gold   : {((~has_rep) & (nn.sum(axis=1)==0)).sum():,}  <-- unusable for supervision")

### Interpreting Q1- **`gold AND report`** is your extractor's supervised set. Split it into extractor-train / extractor-val. This is how you *measure* whether your report parsing works instead of guessing.- **`report, NO gold`** is what the extractor is for — the volume play.- **`NO report, no gold`** studies are only usable for self-supervised pretraining, if at all.

## Q2 — Per-label prevalenceThe metric is **macro**-AUC: all twelve labels weigh the same. Your final rank is decided by whichever labels are rarest, because those are where AUC is lowest and noisiest.This table tells you where to spend weeks 6–7. Anything under ~3% prevalence is a problem child.

In [ ]:
gold = train[nn.sum(axis=1) == len(label_cols)].copy()print(f"Computing prevalence on {len(gold):,} fully-labelled studies\n")rows = []for c in label_cols:    v = pd.to_numeric(gold[c], errors="coerce")    pos = int((v == 1).sum())    rows.append({"label": c, "positives": pos, "prevalence": pos/len(gold) if len(gold) else np.nan})prev = pd.DataFrame(rows).sort_values("prevalence")prev["prevalence"] = prev["prevalence"].map(lambda x: f"{x:.2%}")print(prev.to_string(index=False))print()# Rough guide: AUC standard error scales ~ 1/sqrt(n_positives).print("Approx CV noise per label (5-fold, ~1/sqrt(n_pos) on the held-out fold):")for r in sorted(rows, key=lambda r: r["positives"]):    n_fold = r["positives"] / 5    se = 1/np.sqrt(max(n_fold, 1))    flag = "  <-- HIGH VARIANCE" if n_fold < 30 else ""    print(f"  {r['label']:<20} {r['positives']:>5} pos  (~{n_fold:.0f}/fold)  se~{se:.3f}{flag}")

In [ ]:
# Co-occurrence: do findings travel together? If so, multi-task learning helps and# a shared trunk with 12 heads is well motivated.g = gold[label_cols].apply(pd.to_numeric, errors="coerce").fillna(0)print("Label correlation (Pearson):")print(g.corr().round(2).to_string())print()print("Findings per study:")print(g.sum(axis=1).value_counts().sort_index().to_string())

## Q3 & Q4 — What survives in the DICOM metadata?86 tags were kept after de-identification. Two of them would be very valuable:- **Institution / site** — there are 16 sites with different scanners and protocols. If a model can identify the site it can shortcut to site-specific priors, which inflates random-split CV and then collapses on the private LB. If site is recoverable, use **grouped CV**. If not, use scanner/manufacturer as a proxy.- **Laterality (left vs right knee)** — this is critical. Six of the twelve labels are **medial/lateral specific** (`Medial Meniscus`, `Lateral Meniscus`, `Medial OA`, `Lateral OA`). Medial and lateral swap sides between a left and a right knee. That means a horizontal-flip augmentation is **not label-preserving** unless you also swap those label pairs — and it means the model must know which knee it's looking at.Get this wrong and you silently cap your score on four labels.

In [ ]:
import pydicomfrom pydicom.uid import UIDseries_dirs = sorted(glob.glob(f"{ROOT}/train_series/*/*"))print(f"total training series folders: {len(series_dirs):,}")SAMPLE = 120rng = np.random.default_rng(0)pick = rng.choice(len(series_dirs), size=min(SAMPLE, len(series_dirs)), replace=False)tag_values = collections.defaultdict(set)tag_seen   = collections.Counter()tsyntax    = collections.Counter()t0 = time.time()for i in pick:    d = series_dirs[i]    fs = os.listdir(d)    if not fs: continue    try:        ds = pydicom.dcmread(os.path.join(d, fs[0]), stop_before_pixels=True)    except Exception as e:        print("read fail:", e); continue    try:        tsyntax[str(ds.file_meta.TransferSyntaxUID)] += 1    except Exception:        tsyntax["<missing>"] += 1    for el in ds:        if el.keyword in ("PixelData",) or not el.keyword: continue        tag_seen[el.keyword] += 1        try:            v = str(el.value)[:60]            if len(tag_values[el.keyword]) < 12: tag_values[el.keyword].add(v)        except Exception:            passprint(f"scanned {len(pick)} series in {time.time()-t0:.1f}s")print(f"distinct tags present: {len(tag_seen)}\n")print("=== ALL TAGS (name, how many series had it, distinct values seen) ===")for k, n in sorted(tag_seen.items()):    vals = tag_values[k]    preview = " | ".join(sorted(vals)[:3])    print(f"{k:<34} {n:>4}  [{len(vals)}]  {preview[:80]}")

In [ ]:
print("=== TRANSFER SYNTAXES (drives decode cost) ===")for k, v in tsyntax.most_common():    try: name = UID(k).name    except Exception: name = "?"    print(f"  {v:>4}  {k}  {name}")print()print("=== HUNTING FOR SITE / LATERALITY / SCANNER ===")INTEREST = ["InstitutionName","InstitutionAddress","StationName","Manufacturer",            "ManufacturerModelName","MagneticFieldStrength","DeviceSerialNumber",            "SoftwareVersions","Laterality","ImageLaterality","BodyPartExamined",            "SeriesDescription","ProtocolName","StudyDescription","PatientPosition",            "ImageOrientationPatient","SeriesNumber","ScanningSequence","SequenceName",            "SliceThickness","PixelSpacing","Rows","Columns"]for k in INTEREST:    if k in tag_seen:        vals = sorted(tag_values[k])        print(f"\n{k}  (present in {tag_seen[k]}/{len(pick)} series, {len(vals)} distinct):")        for v in vals[:10]: print(f"     {v}")    else:        print(f"\n{k}: ABSENT (stripped)")

### How to read Q3/Q4- If `InstitutionName` / `StationName` / `DeviceSerialNumber` survived → build grouped CV on it immediately.- If they're stripped, the tuple `(Manufacturer, ManufacturerModelName, MagneticFieldStrength, PixelSpacing)` is usually a decent site fingerprint. Cluster on it.- If `Laterality` / `ImageLaterality` survived → you can flip all knees to a canonical side, which is a genuine free win: it removes a nuisance factor and makes medial/lateral labels consistent.- If laterality is stripped, check `ImageOrientationPatient` — the patient-coordinate direction cosines let you derive left vs right geometrically. Failing that, **disable horizontal flip augmentation entirely** rather than corrupting four labels.- `SeriesDescription` / `ProtocolName` are often more reliable than the provided `Fluid_Sensitive` / `Fat_Suppression` flags, and they're free extra features. But they're multilingual and site-specific, so normalize them.

## Decode benchmark — sizing the caching pipelineThe plan calls for pre-decoding every DICOM once on CPU and storing a resized uint8 cache. This measures how long that will take so you can decide the resolution and whether to shard the job across several notebook sessions.Kaggle CPU notebooks give you 4 cores and a 12-hour session cap, and they **don't consume GPU quota** — which is the whole point.

In [ ]:
import cv2def decode_series(d, n_max=None):    fs = sorted(os.listdir(d))    if n_max: fs = fs[:n_max]    out = []    for f in fs:        ds = pydicom.dcmread(os.path.join(d, f))        out.append(ds.pixel_array)    return outbench = [series_dirs[i] for i in pick[:15]]t0 = time.time(); total = 0for d in bench:    try:        arrs = decode_series(d)        total += len(arrs)    except Exception as e:        print("decode fail:", d, e)el = time.time() - t0rate = total/el if el else 0print(f"decoded {total} slices from {len(bench)} series in {el:.1f}s  ->  {rate:.1f} slices/sec (1 core)")print()n_series_total = len(series_dirs)avg_slices = total/max(len(bench),1)est_slices = n_series_total * avg_slicesprint(f"avg slices/series      : {avg_slices:.1f}")print(f"est. total train slices: {est_slices:,.0f}")print()for cores in (1, 4):    hrs = est_slices / (rate*cores) / 3600 if rate else float('nan')    print(f"full cache pass @ {cores} core(s): {hrs:.1f} h" + ("   <-- shard across sessions" if hrs > 10 else ""))print()for px in (128, 160, 224):    gb = est_slices * px * px / 1e9    print(f"cache size @ {px}px uint8: {gb:.1f} GB")

## Q5 — Report structure and languageDetermines the extraction approach. Structured reports with a `FINDINGS:` / `IMPRESSION:` split are much easier: the impression carries the conclusions, and you can weight it higher.Also watch for **negation and hedging** — "no evidence of meniscal tear" and "possible partial ACL tear" are the two failure modes that will wreck a naive keyword extractor.

In [ ]:
if "Report" in train.columns:    rep = train["Report"].dropna().astype(str)    rep = rep[rep.str.strip() != ""]    print(f"reports: {len(rep):,}")    L = rep.str.len()    print(f"length chars: min={L.min()} p25={L.quantile(.25):.0f} median={L.median():.0f} p75={L.quantile(.75):.0f} max={L.max()}")    print(f"length words: median={rep.str.split().str.len().median():.0f}")    print()    import re    HEADERS = ["FINDINGS","IMPRESSION","CONCLUSION","TECHNIQUE","COMPARISON","HISTORY",               "HALLAZGOS","IMPRESION","CONCLUSAO","ACHADOS","BEFUND","BEURTEILUNG",               "CONCLUSIONE","REPERTI","CONCLUSION:","DIAGNOSTIC"]    print("=== section-header hit rate ===")    up = rep.str.upper()    for h in HEADERS:        c = up.str.contains(h, regex=False).sum()        if c: print(f"  {h:<14} {c:>6}  ({c/len(rep):.1%})")    print()    print("=== NEGATION / HEDGING markers (the classic extractor killers) ===")    for pat in [r"\bno\b", r"\bwithout\b", r"\bnegative\b", r"\bnormal\b", r"\bintact\b",                r"\bpossible\b", r"\bsuspect", r"\bcannot exclude\b", r"\bquestionable\b",                r"\bmild\b", r"\bmoderate\b", r"\bsevere\b"]:        c = rep.str.contains(pat, case=False, regex=True).sum()        print(f"  {pat:<22} {c:>6}  ({c/len(rep):.1%})")else:    print("No Report column found.")

In [ ]:
# Language mix. The RSNA site says nine languages across 16 institutions.try:    from langdetect import detect, DetectorFactory    DetectorFactory.seed = 0    have = Trueexcept ImportError:    print("langdetect not installed -> !pip install langdetect  (internet ON, this is a training notebook)")    have = Falseif have and "Report" in train.columns:    samp = rep.sample(min(1200, len(rep)), random_state=0)    langs = []    for r in samp:        try: langs.append(detect(r[:800]))        except Exception: langs.append("??")    vc = pd.Series(langs).value_counts()    print("language mix (sampled):")    print((vc/len(samp)).map(lambda x: f"{x:.1%}").to_string())

In [ ]:
# Eyeball a few reports per language. Do this. Ten minutes of reading raw reports# is worth more than a day of tuning -- you will immediately see how findings are# phrased, whether OA is graded, and how negation is expressed.if "Report" in train.columns:    for r in rep.sample(4, random_state=1):        print("="*100)        print(r[:1600])    print("="*100)

## Summary — what to decide from this| Finding | Decision it drives ||---|---|| Gold set is large (>1500) | Train supervised first, add pseudo-labels as augmentation || Gold set is small (<800) | Report extraction is the whole game — spend weeks 2–3 there || Any label under ~3% prevalence | Class-balanced sampling + per-class loss weights; expect noisy CV || Site recoverable | Use grouped CV — otherwise your CV is lying to you || Laterality recoverable | Canonicalize all knees to one side; flip augmentation becomes safe || Laterality NOT recoverable | Disable horizontal flips, or swap medial/lateral label pairs when flipping || Decode >10 h single-core | Shard the caching job across multiple CPU sessions || Reports structured | Weight the impression section; consider extracting from it alone || Heavy negation/hedging | Skip lexicons, go straight to LLM or fine-tuned XLM-R extraction |